In [11]:
import pandas as pd
import numpy as np

df = pd.read_excel("basedados.xlsx")

dataframe_dados_clientes = df.iloc[:, 1:19]
dataframe_gabarito       = df.iloc[:, 19]

array_dados_clientes = dataframe_dados_clientes.values
array_gabarito       = dataframe_gabarito.values


In [12]:
def criar_cromossomos(qtd_cromossomos: int = 6, qtd_genes: int = 19) -> np.ndarray:
    return -1 + 2 * np.random.rand(qtd_cromossomos, qtd_genes)

In [13]:
def calcular_fitness(
    cromossomos: np.ndarray,
    array_dados_clientes: np.ndarray,
    array_gabarito: np.ndarray,
) -> np.ndarray:
    total_adimplentes   = np.sum(array_gabarito == 1)
    total_inadimplentes = np.sum(array_gabarito == 0)

    lista_fitness = []

    for linha in cromossomos:
        bias  = linha[0]
        genes = linha[1:]

        q = np.dot(array_dados_clientes, genes) + bias
        vetor_hipotese = np.where(q >= 0, 1, 0)

        acertos_adimplentes   = np.sum((vetor_hipotese == 1) & (array_gabarito == 1))
        acertos_inadimplentes = np.sum((vetor_hipotese == 0) & (array_gabarito == 0))

        percentual_adimplente   = acertos_adimplentes   / total_adimplentes
        percentual_inadimplente = acertos_inadimplentes / total_inadimplentes

        lista_fitness.append(percentual_adimplente * percentual_inadimplente)

    return np.array(lista_fitness)


def fitness_percentual(vetor_fitnesses: np.ndarray) -> np.ndarray:
    soma = np.sum(vetor_fitnesses)
    if soma == 0:
        return np.ones(len(vetor_fitnesses)) / len(vetor_fitnesses)
    return vetor_fitnesses / soma


In [14]:
def selecionar_pais_roleta(cromossomos, percentual_fitnesses):
    roleta_acumulada = np.cumsum(percentual_fitnesses)

    indice_pai = min(np.searchsorted(roleta_acumulada, np.random.rand()), len(cromossomos) - 1)
    
    indice_mae = indice_pai
    while indice_mae == indice_pai:  # garante pais distintos
        indice_mae = min(np.searchsorted(roleta_acumulada, np.random.rand()), len(cromossomos) - 1)

    return cromossomos[indice_pai], cromossomos[indice_mae]

In [15]:
def cruzar_pais(pai: np.ndarray, mae: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    c1 = np.random.randint(1, len(pai))
    c2 = np.random.randint(1, len(pai))
    c3 = np.random.randint(1, len(pai))

    filho1 = np.concatenate([pai[:c1], mae[c1:]])
    filho2 = np.concatenate([pai[:c2], mae[c2:]])
    filho3 = np.concatenate([pai[:c3], mae[c3:]])

    return filho1, filho2, filho3

In [16]:
def mutar(
    filho1: np.ndarray,
    filho2: np.ndarray,
    filho3: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    for filho in [filho1, filho2, filho3]:
        indice = np.random.randint(0, len(filho))
        filho[indice] = -1 + 2 * np.random.rand()

    return filho1, filho2, filho3

In [17]:
def atualizar_populacao(
    cromossomos: np.ndarray,
    vetor_fitnesses: np.ndarray,
    filho1: np.ndarray,
    filho2: np.ndarray,
    filho3: np.ndarray,
    array_dados_clientes: np.ndarray,
    array_gabarito: np.ndarray,
) -> np.ndarray:
    filhos = np.array([filho1, filho2, filho3])
    fitnesses_filhos = calcular_fitness(filhos, array_dados_clientes, array_gabarito)

    indices_melhores_filhos = np.argsort(fitnesses_filhos)[-2:]
    indices_piores          = np.argsort(vetor_fitnesses)[:2]

    nova_populacao = cromossomos.copy()
    for i, idx_pior in enumerate(indices_piores):
        nova_populacao[idx_pior] = filhos[indices_melhores_filhos[i]]

    return nova_populacao

In [18]:
def algoritmo_genetico(
    array_dados_clientes: np.ndarray,
    array_gabarito: np.ndarray,
    qtd_cromossomos: int = 99,
    qtd_genes: int       = 19,
    geracoes: int        = 10000,
    fitness_alvo: float  = 0.99,
) -> tuple[np.ndarray, float]:

    populacao = criar_cromossomos(qtd_cromossomos, qtd_genes)

    melhor_cromossomo = None
    melhor_fitness    = 0.0

    for geracao in range(geracoes):

        fitnesses   = calcular_fitness(populacao, array_dados_clientes, array_gabarito)
        percentuais = fitness_percentual(fitnesses)

        idx_melhor = np.argmax(fitnesses)
        if fitnesses[idx_melhor] > melhor_fitness:
            melhor_fitness    = fitnesses[idx_melhor]
            melhor_cromossomo = populacao[idx_melhor].copy()

        print(f"Geração {geracao+1:>3} | melhor fitness: {melhor_fitness:.4f}")

        if melhor_fitness >= fitness_alvo:
            print(f"\nFitness alvo {fitness_alvo} atingido na geração {geracao+1}.")
            break

        pai, mae = selecionar_pais_roleta(populacao, percentuais)
        filho1, filho2, filho3 = cruzar_pais(pai, mae)
        filho1, filho2, filho3 = mutar(filho1, filho2, filho3)
        populacao = atualizar_populacao(
            populacao, fitnesses,
            filho1, filho2, filho3,
            array_dados_clientes, array_gabarito,
        )

    return melhor_cromossomo, melhor_fitness


In [19]:
melhor_individuo, fitness_final = algoritmo_genetico(
    array_dados_clientes,
    array_gabarito,
)

print(f"\nMelhor fitness final : {fitness_final:.4f}")
print(f"Melhor cromossomo    :\n{melhor_individuo}")

Geração   1 | melhor fitness: 0.7719
Geração   2 | melhor fitness: 0.7719
Geração   3 | melhor fitness: 0.7719
Geração   4 | melhor fitness: 0.7719
Geração   5 | melhor fitness: 0.7719
Geração   6 | melhor fitness: 0.7719
Geração   7 | melhor fitness: 0.7719
Geração   8 | melhor fitness: 0.7719
Geração   9 | melhor fitness: 0.7719
Geração  10 | melhor fitness: 0.7719
Geração  11 | melhor fitness: 0.7719
Geração  12 | melhor fitness: 0.7719
Geração  13 | melhor fitness: 0.7719
Geração  14 | melhor fitness: 0.7719
Geração  15 | melhor fitness: 0.7719
Geração  16 | melhor fitness: 0.7719
Geração  17 | melhor fitness: 0.7719
Geração  18 | melhor fitness: 0.7719
Geração  19 | melhor fitness: 0.7719
Geração  20 | melhor fitness: 0.7719
Geração  21 | melhor fitness: 0.7719
Geração  22 | melhor fitness: 0.7719
Geração  23 | melhor fitness: 0.7719
Geração  24 | melhor fitness: 0.7719
Geração  25 | melhor fitness: 0.7719
Geração  26 | melhor fitness: 0.7719
Geração  27 | melhor fitness: 0.7719
G

KeyboardInterrupt: 

criando a funcao pra prever

In [21]:
def prever(
    cromossomo: np.ndarray,
    array_dados_novos_clientes: np.ndarray,
) -> np.ndarray:
    """
    Classifica novos clientes usando o melhor cromossomo treinado.
    
    Retorna:
        np.ndarray: 1 = adimplente, 0 = inadimplente
    """
    bias  = cromossomo[0]
    genes = cromossomo[1:]

    q = np.dot(array_dados_novos_clientes, genes) + bias
    return np.where(q >= 0, 1, 0)